In [8]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [9]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)


In [129]:
def robust_counter_powerU (sets,p,R,r,m,r_f,c,rav):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    f_obj = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
        constraints.append(-cp.power(z3,rav)/rav - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(cp.abs(a)<=100)
    constraints.append(alpha + beta + gamma * (r-1) + z4 + z2 <= -c**rav/rav)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

In [136]:
def robustcheckpoweru(a,R,r,p,m,r_f,rav):
    N = len(p)
    x = -(R.dot(a)+(1-sum(a))*r_f)**rav/rav
    rank = np.argsort(-x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,q_b.value)

In [137]:
def squeeze_algo_putility(R,r,c,p,m,r_f,rav):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=100]
    h = np.zeros(N)
    iterations = 1
    steps = 1
    f_obj = 0
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    for i in range(N):
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
    z1 = ((R @ a)+ (1-cp.sum(a))*r_f)
    constraints.append(-h.T@(cp.power(z1,rav)/rav)<= -(c+1e-10)**rav/rav)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    oldrank = np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav)
    sets = ranktoset(oldrank)
    nonstop = True
    while nonstop:
        [rbvalue,h] = robustcheckpoweru(w,R,r,p,m,r_f,rav)
        print('rbvalue',rbvalue)
        if rbvalue <= -(c+1e-10)**rav/rav:
            return('cut-stop',w,upperobj,iterations,steps)
        constraints.append(-h.T@(cp.power(z1,rav)/rav)<= -(c+1e-10)**rav/rav)
        iterations = iterations + 1
        nonstop2 = True
        once = False
        while nonstop2:
            [w,lowerobj] = robust_counter_powerU (sets,p,R,r,m,r_f,c,rav)
            newrank = np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav)
            if np.array_equal(newrank,oldrank):
                break
            once = True
            oldrank = newrank
            [sets,added] = makesetflex(sets, newrank)
            steps = steps + 1
            print('steps',steps)
        if once:
            [rbvalue,h] = robustcheckpoweru(w,R,r,p,m,r_f,rav)
            constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= -(c+1e-10)**rav/rav)
            iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value 
        [sets,added] = makesetflex(sets, np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav))
        print(upperobj, lowerobj, iterations,steps)

In [143]:
def normal_cutting_plane(R,r,c,p,m,r_f,rav):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=100]
    h = np.zeros(N)
    iterations = 1
    f_obj = 0
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    for i in range(N):
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
    z1 = ((R @ a)+ (1-cp.sum(a))*r_f)
    constraints.append(-h.T@(cp.power(z1,rav)/rav)<= -(c+1e-10)**rav/rav)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    nonstop = True
    while nonstop:
        [rbvalue,h] = robustcheckpoweru(w,R,r,p,m,r_f,rav)
        print('rbvalue',rbvalue)
        if rbvalue <= -(c+1e-10)**rav/rav:
            return('cut-stop',w,upperobj,iterations)
        constraints.append(-h.T@(cp.power(z1,rav)/rav)<= -(c+1e-10)**rav/rav)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value 
        print(upperobj, iterations)

In [13]:
np.random.seed(5)

In [14]:
N=30
p = np.zeros(N)+1/N
I = 3
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
#print(R)

[0.07174797 0.07208475 0.06633768]


In [115]:
rav= 1-2
r = 0.3
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.005
c = 0.0045

In [138]:
squeeze_algo_putility(R,r,c,p,m,r_f,rav)

rbvalue 348.0042419466459
steps 2
steps 3
steps 4
steps 5
-189.04958957811718 -189.19515214466077 3 5
rbvalue 241.9845526259498
-189.19235731154805 -189.19604375286005 4 5
rbvalue 224.07562316986773
steps 6
-189.19523125657426 -189.19589057137946 6 6
rbvalue 222.22222534233111
-189.19522290255202 -189.19589057137946 7 6
rbvalue 222.22228146114747
-189.19523292729772 -189.19589057137946 8 6
rbvalue 222.22221862975522
-189.1952333280845 -189.19589057137946 9 6
rbvalue 222.22221464347814


('cut-stop',
 array([0.00179003, 0.00150677, 0.00199619]),
 -189.1952333280845,
 9,
 6)

In [145]:
normal_cutting_plane(R,r,c,p,m,r_f,rav)

rbvalue 348.0042419466459
-189.04958957811718 2
rbvalue 241.9845526259498
-189.19235731154805 3
rbvalue 224.07562316986773
-189.19523125657426 4
rbvalue 222.22222534233111
-189.19522290255202 5
rbvalue 222.22228146114747
-189.19523292729772 6
rbvalue 222.22221862975522
-189.1952333280845 7
rbvalue 222.22221464347814


('cut-stop',
 array([0.00179003, 0.00150677, 0.00199619]),
 -189.1952333280845,
 7)

In [122]:
-(c+1e-10)**rav/rav

222.22221728395076